In [1]:
# ==============================================================================
# IMPROVED ASD DETECTION PIPELINE (MLP + MULTI-ATLAS + TANGENT)
# ==============================================================================

!pip install nilearn scikit-learn pandas numpy torch tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from nilearn.datasets import fetch_abide_pcp, fetch_atlas_schaefer_2018, fetch_atlas_aal
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 87.8 MB/s eta 0:00:00


In [2]:
# -----------------------------
# CONFIG
# -----------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# -----------------------------
# DATA
# -----------------------------
print("Fetching ABIDE...")
abide = fetch_abide_pcp(n_subjects=200, pipeline='cpac')

phenotypic = pd.DataFrame(abide.phenotypic)
y = (phenotypic['DX_GROUP'] == 1).astype(int).values

Fetching ABIDE...


[fetch_abide_pcp] Added README.md to /root/nilearn_data

[fetch_abide_pcp] Dataset created in /root/nilearn_data/ABIDE_pcp

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv ...

[fetch_abide_pcp]  ...done. (1 seconds, 0 min)

...

[fetch_abide_pcp] Downloaded 24387584 of 176080760 bytes (13.9%%,    6.3s remaining)

[fetch_abide_pcp] Downloaded 64659456 of 176080760 bytes (36.7%%,    3.5s remaining)

[fetch_abide_pcp] Downloaded 109420544 of 176080760 bytes (62.1%%,    1.8s remaining)

[fetch_abide_pcp] Downloaded 155901952 of 176080760 bytes (88.5%%,    0.5s remaining)

[fetch_abide_pcp]  ...done. (5 seconds, 0 min)

In [8]:
from tqdm import tqdm
from nilearn.datasets import fetch_atlas_schaefer_2018, fetch_atlas_aal
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
import numpy as np
import requests
from unittest.mock import patch

# -----------------------------
# MULTI-ATLAS FEATURE EXTRACTION
# -----------------------------
# Capture the original send method
original_send = requests.Session.send

def modified_send(self, request, **kwargs):
    kwargs['verify'] = False
    return original_send(self, request, **kwargs)

with patch('requests.Session.send', new=modified_send):
    print("Fetching atlases (SSL verification disabled)... ")
    atlas1 = fetch_atlas_schaefer_2018(n_rois=100)
    atlas2 = fetch_atlas_aal()

masker1 = NiftiLabelsMasker(labels_img=atlas1.maps, standardize="zscore_sample")
masker2 = NiftiLabelsMasker(labels_img=atlas2.maps, standardize="zscore_sample")

ts1, ts2 = [], []

print("Extracting time-series from multi-atlas...")
for img in tqdm(abide.func_preproc):
    ts1.append(masker1.fit_transform(img))
    ts2.append(masker2.fit_transform(img))

conn = ConnectivityMeasure(kind='tangent')

print("Computing connectivity measures...")
X1 = conn.fit_transform(ts1)
X2 = conn.fit_transform(ts2)

X1 = np.array([c.flatten() for c in X1])
X2 = np.array([c.flatten() for c in X2])

# FEATURE FUSION
X = np.concatenate([X1, X2], axis=1)
print(f"Feature extraction complete. Combined feature shape: {X.shape}")

Fetching atlases (SSL verification disabled)... 


[fetch_atlas_schaefer_2018] Dataset found in /root/nilearn_data/schaefer_2018

[fetch_atlas_aal] Dataset found in /root/nilearn_data/aal_3v2

[fetch_atlas_aal] Downloading data from https://www.gin.cnrs.fr/wp-content/uploads/AAL3v2_for_SPM12.tar.gz ...

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.gin.cnrs.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

[fetch_atlas_aal] Extracting data from 
/root/nilearn_data/aal_3v2/43f38da73bc7adb6022df5794d84f2eb/AAL3v2_for_SPM12.tar.gz...

[fetch_atlas_aal] .. done.

Extracting time-series from multi-atlas...


  0%|          | 0/200 [00:00<?, ?it/s]/tmp/ipykernel_2597/371563763.py:32: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.uint8(160), np.uint8(134)}. Label image only contains 165 labels (including background).
  ts2.append(masker2.fit_transform(img))
  0%|          | 1/200 [00:15<51:47, 15.62s/it]/tmp/ipykernel_2597/371563763.py:32: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.uint8(160), np.uint8(134)}. Label image only contains 165 labels (including background).
  ts2.append(masker2.fit_transform(img))
  1%|          | 2/200 [00:28<46:29, 14.09s/it]/tmp/ipykernel_2597/371563763.py:32: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.uint8(160), np.uint8(134)}. Label image only contains 165 labels (including background).
  ts2.append(masker2.fit_transform(img))
  2%|▏         | 3/200 [00:41<44:29, 13.55s/it]/tmp/ip

Computing connectivity measures...
Feature extraction complete. Combined feature shape: (200, 36896)


In [9]:
# -----------------------------
# MODEL
# -----------------------------
class ASDModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [11]:
from nilearn.maskers import NiftiMasker
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA

# -----------------------------
# TRAIN LOOP
# -----------------------------
# Initialize Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
acc_scores, auc_scores = [], []

# Convert X to a numpy array
X_np = np.array(X)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_np, y)):

    print(f"\n--- Fold {fold+1} ---")

    X_train, X_test = X_np[train_idx], X_np[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # SCALE (no leakage)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # PCA (dimensionality reduction)
    pca = PCA(n_components=50)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train).to(DEVICE)
    y_train_t = torch.FloatTensor(y_train).view(-1,1).to(DEVICE)
    X_test_t = torch.FloatTensor(X_test).to(DEVICE)

    # Use the defined ASDModel
    model = ASDModel(X_train.shape[1]).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()

    # Training
    model.train()
    for epoch in range(30):
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(float)

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    acc_scores.append(acc)
    auc_scores.append(auc)

    print(f"Accuracy: {acc:.3f}, AUC: {auc:.3f}")


--- Fold 1 ---
Accuracy: 0.650, AUC: 0.823

--- Fold 2 ---
Accuracy: 0.750, AUC: 0.929

--- Fold 3 ---
Accuracy: 0.650, AUC: 0.737

--- Fold 4 ---
Accuracy: 0.600, AUC: 0.616

--- Fold 5 ---
Accuracy: 0.850, AUC: 0.909

--- Fold 6 ---
Accuracy: 0.800, AUC: 0.838

--- Fold 7 ---
Accuracy: 0.500, AUC: 0.606

--- Fold 8 ---
Accuracy: 0.800, AUC: 0.788

--- Fold 9 ---
Accuracy: 0.650, AUC: 0.727

--- Fold 10 ---
Accuracy: 0.750, AUC: 0.859


In [12]:
# -----------------------------
# CROSS VALIDATION
# -----------------------------
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

acc_scores, auc_scores = [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y)):

    print(f"\n--- Fold {fold+1} ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # SCALE
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # DATA LOADER
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train).view(-1,1)),
        batch_size=16,
        shuffle=True
    )

    # MODEL
    model = ASDModel(X_train.shape[1]).to(DEVICE)

    pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    pos_weight = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-4)

    # TRAIN
    for epoch in range(40):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # EVAL
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_test).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()

    # THRESHOLD TUNING
    fpr, tpr, thresholds = roc_curve(y_test, probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]

    preds = (probs > best_thresh).astype(int)

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    acc_scores.append(acc)
    auc_scores.append(auc)

    print(f"Accuracy: {acc:.3f}, AUC: {auc:.3f}")


--- Fold 1 ---
Accuracy: 0.850, AUC: 0.854

--- Fold 2 ---
Accuracy: 0.900, AUC: 0.970

--- Fold 3 ---
Accuracy: 0.750, AUC: 0.818

--- Fold 4 ---
Accuracy: 0.650, AUC: 0.737

--- Fold 5 ---
Accuracy: 0.800, AUC: 0.859

--- Fold 6 ---
Accuracy: 0.750, AUC: 0.838

--- Fold 7 ---
Accuracy: 0.600, AUC: 0.616

--- Fold 8 ---
Accuracy: 0.700, AUC: 0.778

--- Fold 9 ---
Accuracy: 0.800, AUC: 0.838

--- Fold 10 ---
Accuracy: 0.800, AUC: 0.869


In [13]:
# FINAL
print("\n===== FINAL RESULTS =====")
print(f"Mean Accuracy: {np.mean(acc_scores)*100:.2f}%")
print(f"Mean AUC: {np.mean(auc_scores):.3f}")


===== FINAL RESULTS =====
Mean Accuracy: 76.00%
Mean AUC: 0.818
